# Day 4 · GitHub PR 리뷰를 dry-run과 사람 승인으로 보호하기

화면을 따라 실행하되, 결과를 자동 게시하지 않습니다. 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
# 최초 1회 설치. 이미 설치했다면 빠르게 완료됩니다.
%pip install -q -r ../../requirements-day1.txt
# STT 실습을 실제 음성으로 실행할 때만 다음 줄의 주석을 해제합니다.
# %pip install -q -r ../../requirements-stt-optional.txt

In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": str(ROOT), "python": sys.version.split()[0]})

## 1. synthetic PR fixture를 읽고 target을 고정합니다

In [ ]:
from src.course_services.github_service import (
    InMemoryIdempotencyStore, load_pr_fixture,
    prepare_review_comment, publish_review_comment,
)
from src.course_services.review_service import run_review_service

fixture = load_pr_fixture(ROOT / "data/day4_github/pr_fixture.json", workspace_root=ROOT)
diff_text = (ROOT / fixture["diff"]).read_text(encoding="utf-8")
report = run_review_service(diff_text)
target = {key: fixture[key] for key in ("repository", "number", "head_sha")}
print(target)

## 2. 첫 실행은 무조건 dry-run payload입니다

In [ ]:
dry_run_plan = prepare_review_comment(report=report, target=target, dry_run=True)
print(json.dumps(dry_run_plan, ensure_ascii=False, indent=2))
assert dry_run_plan["external_write"] is False

## 3. 수업에서는 fake publisher로 승인과 중복 방지를 검증합니다

In [ ]:
approval_plan = prepare_review_comment(report=report, target=target, dry_run=False)
store = InMemoryIdempotencyStore()
calls = []

def fake_publisher(pr_target, body):
    calls.append({"target": pr_target.model_dump(), "body": body})
    return {"id": 101, "url": "https://example.invalid/reviews/101"}

first = publish_review_comment(
    plan=approval_plan, human_approved=True, publisher=fake_publisher, store=store,
)
second = publish_review_comment(
    plan=approval_plan, human_approved=True, publisher=fake_publisher, store=store,
)
print(json.dumps({"first": first, "second": second, "call_count": len(calls)}, ensure_ascii=False, indent=2))
assert len(calls) == 1

## 완료 확인

- Day 4 결과 JSON을 확인했습니다.
- 실패 경로가 traceback 대신 `error_code`로 남는지 확인했습니다.
- 외부 쓰기와 자동 메일이 발생하지 않았음을 확인했습니다.
- 변경한 코드는 diff와 test 결과를 사람이 검토합니다.